# Notebook 33 — Reliability, Observability, and Load Testing

    ## Learning objectives

    - Instrument latency, tokens, errors, quality, and cost with safe metadata
- Implement bounded concurrency, retries, cancellation, and backpressure
- Design load tests and deployment regression gates

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 33.1 Observe the full request lifecycle

Correlate request ID, trace ID, tenant-safe identifier, model/revision, prompt version,
decoding config, queue time, TTFT, generation time, input/output tokens, finish reason,
retry count, tool/retrieval spans, validation result, and sampled quality outcome. Do not
log raw prompts by default; apply redaction, retention, access control, and sampling.


In [ ]:
import json, time, uuid
def event(name, **fields):
    record = {"event": name, "timestamp": time.time(), **fields}
    print(json.dumps(record, separators=(",", ":")))

request_id = str(uuid.uuid4())
event("llm.request.start", request_id=request_id, model="configured-model", prompt_version="v3")
event("llm.request.end", request_id=request_id, latency_ms=231, input_tokens=120,
      output_tokens=48, finish_reason="stop", validated=True)


In [ ]:
import asyncio, random
semaphore = asyncio.Semaphore(4)

async def bounded_call(item, attempts=3):
    async with semaphore:
        for attempt in range(attempts):
            try:
                # Replace with an async inference call carrying a timeout.
                await asyncio.sleep(0.01)
                if random.random() < 0.2: raise TimeoutError("simulated transient")
                return {"item": item, "ok": True, "attempt": attempt + 1}
            except TimeoutError:
                if attempt + 1 == attempts: return {"item": item, "ok": False}
                await asyncio.sleep(0.02 * 2**attempt * random.uniform(.5, 1.5))

results = await asyncio.gather(*(bounded_call(i) for i in range(12)))
print(results)


## 33.2 Failure policy

Retry only transient, idempotent operations; respect server retry hints; add jitter; cap
attempts and total deadline. Bound queues and concurrency—unbounded retries amplify
overload. Define fallback behavior and test whether quality remains acceptable. Propagate
cancellation to streams and tools. Use circuit breakers carefully: they protect systems
but can synchronize failures without randomized recovery.


## 33.3 Load and regression testing

Replay realistic distributions of prompt length, output length, streaming, retrieval,
and tool use. Warm the system, increase offered load, and report p50/p95/p99 TTFT and
end-to-end latency, tokens/sec, queue time, timeout/error rate, saturation, and quality.
Distinguish open-loop arrival load from closed-loop user simulation. Gate deployments on
correctness slices plus latency/cost budgets; canary and retain rollback.


## 33.4 Latency and throughput decomposition

End-to-end latency includes client/network, admission queue, tokenization/rendering, retrieval/
tools, prefill, decode, validation, and streaming transport. TTFT includes everything until the
first visible delta. Time per output token/inter-token latency describes decode smoothness. Report
p50/p95/p99 by prompt/output length and workload class; averages hide tail behavior. Throughput
can mean requests/sec or input/output/total tokens/sec—state which.

Little's Law relates average concurrency \(L\), arrival rate \(\lambda\), and time in system
\(W\): \(L=\lambda W\) under stable conditions. As utilization nears capacity, queueing rises
sharply. Maximize sustainable throughput subject to latency/error/quality SLO, not benchmark peak.
Separate cold start/model load from warm requests and distinguish provider-reported usage from
locally estimated tokens.


In [ ]:
# Summarize a synthetic latency trace by percentile and stage.
import numpy as np
trace_rows = [
    {"queue":10,"prefill":80,"decode":240}, {"queue":15,"prefill":90,"decode":260},
    {"queue":180,"prefill":100,"decode":310}, {"queue":12,"prefill":75,"decode":220},
    {"queue":400,"prefill":120,"decode":360},
]
total = np.array([sum(r.values()) for r in trace_rows])
for p in [50, 95, 99]: print(f"p{p} total={np.percentile(total,p):.1f} ms")
print("mean stage ms:", {k: np.mean([r[k] for r in trace_rows]) for k in trace_rows[0]})


## 33.5 Resilience patterns and their boundaries

Set a total deadline, then allocate stage timeouts; otherwise retries can exceed the user's
budget. Retry transient rate-limit/network/5xx failures only when safe. Respect retry-after, use
exponential backoff with jitter, cap attempts, and preserve idempotency. Hedged requests may lower
tails but multiply cost/load and need cancellation. Circuit breakers prevent hammering a failing
dependency but require careful half-open recovery. Bulkheads isolate workloads so one tenant or
slow tool cannot exhaust all concurrency.

Backpressure begins before overload: bounded admission queues, per-tenant quotas, concurrency and
token budgets, and load shedding by priority. Streaming disconnects/cancellations should propagate
upstream. Fallbacks are product behavior, not merely infrastructure: switching to a smaller model,
cached answer, retrieval-only response, or abstention must meet quality/safety evals and be visible
in traces. Never retry deterministic validation or authorization failures.


In [ ]:
# Deadline budgeting helper: remaining time shrinks across stages/retries.
import time
class Deadline:
    def __init__(self, seconds): self.ends = time.monotonic() + seconds
    def remaining(self): return max(0.0, self.ends - time.monotonic())
    def timeout(self, stage_cap):
        value = min(stage_cap, self.remaining())
        if value <= 0: raise TimeoutError("request deadline exhausted")
        return value
deadline = Deadline(2.0)
print("retrieval timeout", deadline.timeout(.4))
time.sleep(.02)
print("generation timeout", deadline.timeout(1.5))


## 33.6 Observability, SLOs, and privacy

Metrics aggregate health; logs capture discrete events; traces connect stages; profiles explain
resource time. Use low-cardinality metric labels—never user IDs or prompts. Trace IDs correlate
request/model/retrieval/tool spans. Capture model/revision, prompt version (not necessarily text),
decoding, token usage, cache, finish reason, validation, retries/fallback, and sampled quality.
Redact at collection, restrict access, encrypt, set retention, and give tenants deletion controls.

Define SLIs and SLOs by user experience: successful validated responses under a latency bound,
availability excluding invalid requests, quality on sampled labeled traffic, and budget. Error
budgets govern release pace. Alert on actionable symptoms and burn rates rather than every transient
error. Dashboards slice by model/version/route/length/tenant tier while controlling cardinality.
Correlate quality with latency/cost; a fast invalid response is not successful.


In [ ]:
# Error-budget calculation for a 99.5% monthly availability SLO.
minutes = 30 * 24 * 60
slo = .995
budget_minutes = minutes * (1-slo)
consumed = 95
print(f"monthly error budget: {budget_minutes:.1f} min")
print(f"consumed: {consumed/budget_minutes:.1%}; remaining: {budget_minutes-consumed:.1f} min")


## 33.7 Load testing and release engineering reference

Model prompt/output distributions, arrival process, streaming, cancellations, retrieval/tools,
cache hit rates, tenants, and failures. Open-loop tests send arrivals independently and expose
queueing; closed-loop clients wait for responses and can hide overload through coordinated
omission. Warm up kernels/caches, run long enough for steady state, and record hardware/software.
Find the knee where tail latency or errors accelerate, then operate with headroom.

Release artifacts pin code, model, tokenizer/template, prompt, index, config, and dependencies.
Run offline evals and load tests; deploy canary; compare quality/latency/cost/safety; expand
gradually; retain automatic/manual rollback. Schema migrations and cache compatibility need plans.
Test cold start, dependency outage, rate limits, corrupt responses, cancellation, deploy during
load, and rollback—not only happy-path throughput.

Capacity plan in tokens: offered input/output tokens/sec, concurrency, KV-cache demand, replicas,
autoscaling lag, quotas, and cost. Autoscaling on CPU alone misses GPU/KV/queue saturation.


## 33.8 Production-readiness reference

| Area | Minimum evidence |
|---|---|
| Quality | Frozen evals, critical slices, canary comparison |
| Latency | Stage spans and p50/p95/p99 under representative load |
| Capacity | Sustainable tokens/sec, queues, headroom, autoscaling lag |
| Reliability | Deadlines, bounded retries/queues, cancellation, fallback tests |
| Observability | Metrics/logs/traces with versions and privacy policy |
| Security | Authn/z, limits, secrets, isolation, incident controls |
| Release | Immutable artifacts, staged rollout, rollback exercise |

An SLO should count only valid useful responses as success. Track error-budget burn across dependency,
validation, timeout, and quality failures. Avoid high-cardinality metrics and sensitive prompt labels.
Propagate request/trace IDs through retrieval, tools, model, and validation.

Load tests must avoid coordinated omission, include prompt/output distributions and cancellations, and
run long enough for queues/autoscaling. Capacity conclusions are invalid without hardware/software and
quality/length controls.


## Exercises

    1. Add a total deadline that covers queueing, retries, generation, and tools.
2. Produce a load-test matrix across prompt lengths and concurrency levels.
3. Design a privacy-preserving trace sampling and retention policy.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
